# P39 — Redes generativas adversarias

## 1. Título y paper

**Paper:** *Generative Adversarial Networks*  
**Autoría:** Ian J. Goodfellow, Jean Pouget-Abadie, Mehdi Mirza, y otros  
**Año y venue:** 2014 · arXiv:1406.2661 · NeurIPS 2014  
**Nivel:** L3 · **Motor:** `gan`  
**Ficha completa:** [`P39_gan`](../../papers/foundational/P39_gan/README.md)

**Hito:** Convierte la generación en un juego: dos redes compiten y ninguna necesita una verosimilitud explícita.

- [arXiv:1406.2661](https://arxiv.org/abs/1406.2661)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los modelos generativos exigían definir y optimizar una verosimilitud, lo que obligaba a aproximaciones costosas o producía muestras borrosas.
2. Ejecutar una implementación mínima de la propuesta: Entrenar un generador contra un discriminador en un juego minimax: el generador gana cuando el discriminador ya no distingue lo real de lo sintético.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P38
- P02


## 4. Intuición

Un falsificador y un policía que aprenden a la vez. El falsificador mejora porque el policía lo pilla; el policía mejora porque el falsificador se refina. Nadie le enseña al falsificador qué es un billete bueno: solo si coló o no.


## 5. Concepto mínimo

```text
min_G max_D  E_x[log D(x)] + E_z[log(1 − D(G(z)))]

    D quiere acertar quién es real     → maximiza
    G quiere que D se equivoque        → minimiza
```

No hay verosimilitud explícita: la señal de entrenamiento la produce **otra red**.


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('gan', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Tiene que cubrir el generador los tres modos de la distribución real para engañar al discriminador?
2. ¿Qué pasará con la diversidad de sus muestras?
3. ¿Cómo lo detectarías si solo miras muestras individuales?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('gan', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('gan', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El generador converge a **un solo modo** de los tres. Y desde el punto de vista de su objetivo, hace bien: engañar al discriminador no exige cubrir la distribución, basta con ser convincente en una región. Eso es el colapso de modos.


## 10. Comentario pedagógico

El colapso es difícil de detectar mirando muestras: cada una puede ser excelente. Hay que medir **cobertura**, no solo calidad. Es el mismo error que en RAG con las citas: la muestra individual se ve bien y el sistema está roto.


## 11. Error o anti-patrón deliberado

Anti-patrón: evaluar un modelo generativo enseñando las mejores muestras.


In [ ]:
print('20 imagenes preciosas elegidas a mano NO son evidencia de nada.')
print('Un generador colapsado produce muestras excelentes... todas parecidas.')
print('Sin medida de cobertura/diversidad, la evaluacion es publicidad.')

## 12. Corrección

Lo mínimo que hay que reportar en un modelo generativo:


In [ ]:
reporte = {'calidad': 'metrica perceptual sobre muestras no elegidas',
           'diversidad': 'cobertura de los modos de la distribucion real',
           'muestras': 'aleatorias con semilla, no seleccionadas',
           'estabilidad': 'varias semillas de entrenamiento, no una'}
show(reporte)

## 13. Desafío guiado

Cambia la posición inicial del generador y observa a qué modo colapsa: siempre al más cercano.


In [ ]:
r = run_paper_lab('gan', seed=3)['result']
show(r)

## 14. Desafío autónomo

Entrena una GAN pequeña sobre una mezcla de gaussianas 2D y mide cuántos modos cubre a lo largo del entrenamiento. Compara con un VAE y con difusión sobre los mismos datos.


## 15. Evidencia de aprendizaje

Guarda la trayectoria del generador, el conteo de modos cubiertos y tu lista de lo que hay que reportar en un modelo generativo.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P39_gan/README.md) · evaluación formal: [`assessments/papers/P39_gan.md`](../../assessments/papers/P39_gan.md)


## 16. Cierre

Generar ya es posible por dos vías. Ahora el andamiaje: qué hace que una red profunda se pueda entrenar sin memorizar.


## 17. Conexión con el siguiente hito

- P17
- modelos generativos de imagen

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
